In [159]:
import pandas as pd

df = pd.read_json("dataset.jsonl", lines = True, nrows = 100)
df = df[["rating", "title", "text"]]

In [160]:
prompt = "I'm going to give you a list of reviews, each with 3 columns: rating, title and text." \
" Based on these 3 parameters only, I want you to determine, for that review, the sentiment of it. Keep" \
" The output should be in a CSV format and you should just answer with the CSV content, nothing else." \
"Your response should" \
" just be CSV content. The output should be a 2D table: One row per review, and then 3 columns for 3 different" \
" sentiments. If one review only expresses one sentiment, leave the cell for the other 2 sentiments empty. Do not" \
" use over 20 unique sentiments. Do not include the rating, title and text columns" \
" in your output. Columns should be named s1, s2 and s3. The values of s1," \
" s2 and s3 should be strings that represent sentiments, such as 'surprise', 'disappointment', 'excited'." \
" This is the CSV content for the reviews:\n\n"

csv = df.to_csv()
prompt += csv

prompt += "\n Remember the output: One row per review. 3 Columns: s1, s2, s3. Each column's values" \
"can contain different sentiments. Avoid generic sentiments such as positive, negative or neutral. The first line" \
"should state the column names: index, s1, s2, s3. DO NOT include an index column. The first column should be s1." \
" Do not start every line with a comma."

In [161]:
from dotenv import load_dotenv
from google import genai
import os

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
response = client.models.generate_content(
    model="gemini-flash-latest",  # comprova el nom actual a AI Studio
    contents=prompt,
).text

response = response.replace("```csv", "").replace("```", "").strip()

with open("response.csv", "w") as file:
    file.write(response)

In [162]:
output = pd.read_csv("response.csv")
print(output)
print(set(output["s1"]))

              s1            s2          s3
0          anger      distrust         NaN
1      enjoyment  satisfaction         NaN
2     enthusiasm       delight         NaN
3   satisfaction     enjoyment         NaN
4      enjoyment       delight         NaN
..           ...           ...         ...
95  indifference  satisfaction         NaN
96  satisfaction  indifference         NaN
97     enjoyment  indifference         NaN
98  satisfaction  indifference         NaN
99       delight  satisfaction  enthusiasm

[100 rows x 3 columns]
{'boredom', 'indifference', 'amusement', 'frustration', 'disappointment', 'annoyance', 'surprise', 'delight', 'anger', 'satisfaction', 'enthusiasm', 'enjoyment'}
